# B4: Generic I-JEPA Full Fine-Tune

**Leaf-JEPA IRP** | Stage 3 — Baseline Establishment

## Rationale

B4 is the **upper-bound ceiling**. All 632M parameters of the I-JEPA encoder are unfrozen
and fine-tuned end-to-end alongside a classification head. This is the most expensive
baseline — it answers: what's the best this architecture can achieve with unlimited compute?

For **RQ2**: PEFT methods should approach B4's performance with <2% of its trainable parameters.

**Warning**: This is the slowest notebook. Use AMP, gradient checkpointing, and gradient
accumulation. Expect 2-4 hours on A100, 6-12 hours on T4.

## Initialization

In [1]:
# Setup and hyperparameter documentation
import sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from stage3_baseline_establishment.config_stage3 import *
from stage3_baseline_establishment.baseline_utils import (
    seed_everything, PlantVillageDataset, load_split,
    train_one_epoch, evaluate, EarlyStopping, save_results,
    plot_confusion_matrix, load_ijepa_encoder
)

from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

verify_config()
seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hyperparams = {
    "baseline": "B4",
    "model": "I-JEPA ViT-H/14 Full Fine-Tune",
    "total_params": "~632M (ALL UNFROZEN)",
    "backbone_lr": FT_LR,
    "head_lr": FT_HEAD_LR,
    "weight_decay": FT_WEIGHT_DECAY,
    "batch_size": FT_BATCH_SIZE,
    "grad_accum": FT_GRAD_ACCUM,
    "effective_batch": FT_BATCH_SIZE * FT_GRAD_ACCUM,
    "max_epochs": FT_EPOCHS,
    "patience": FT_PATIENCE,
    "lr_decay_factor": FT_LR_DECAY,
    "gradient_checkpointing": FT_GRAD_CKPT,
    "amp": True,
}
BASELINES_DIR.mkdir(parents=True, exist_ok=True)
save_results(hyperparams, BASELINES_DIR / "B4_hyperparams.json")

  ✅  All config checks passed.
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\baselines\B4_hyperparams.json


## Build model

In [2]:
# Build model with layerwise LR decay
import wandb

print("Loading I-JEPA encoder for full fine-tuning...")
encoder = load_ijepa_encoder(IJEPA_CHECKPOINT, VIT_MODEL_NAME, device)

# Enable gradient checkpointing if configured
if FT_GRAD_CKPT and hasattr(encoder, "set_grad_checkpointing"):
    encoder.set_grad_checkpointing(enable=True)
    print("  Gradient checkpointing enabled")

# Build full model: encoder + classification head
class IJEPAClassifier(nn.Module):
    def __init__(self, encoder, num_classes, embed_dim):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        features = self.encoder(x)
        return self.head(features)

model = IJEPAClassifier(encoder, NUM_CLASSES, EMBED_DIM).to(device)

# Layerwise LR decay
def get_layerwise_params(model, base_lr, head_lr, decay_factor):
    param_groups = []

    # Head — highest LR
    param_groups.append({
        "params": list(model.head.parameters()),
        "lr": head_lr,
        "name": "head"
    })

    # Encoder blocks — decaying LR from top to bottom
    if hasattr(model.encoder, "blocks"):
        n_blocks = len(model.encoder.blocks)
        for i, block in enumerate(model.encoder.blocks):
            layer_lr = base_lr * (decay_factor ** (n_blocks - 1 - i))
            param_groups.append({
                "params": list(block.parameters()),
                "lr": layer_lr,
                "name": f"block_{i}"
            })
    else:
        # Fallback: all encoder params at base_lr
        param_groups.append({
            "params": [p for p in model.encoder.parameters()],
            "lr": base_lr,
            "name": "encoder"
        })

    # Remaining encoder params (patch embed, norm, etc.)
    block_param_ids = set()
    for pg in param_groups:
        for p in pg["params"]:
            block_param_ids.add(id(p))
    remaining = [p for p in model.encoder.parameters() if id(p) not in block_param_ids]
    if remaining:
        param_groups.append({
            "params": remaining,
            "lr": base_lr,
            "name": "encoder_other"
        })

    return param_groups

param_groups = get_layerwise_params(model, FT_LR, FT_HEAD_LR, FT_LR_DECAY)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total trainable params: {n_trainable:,}")
for pg in param_groups:
    n = sum(p.numel() for p in pg["params"])
    print(f"    {pg['name']:>15}: {n:>12,} params, lr={pg['lr']:.2e}")

Loading I-JEPA encoder for full fine-tuning...
  Loaded checkpoint: IN1K-vit.h.14-300e.pth.tar
  Missing keys:  1
  Unexpected keys: 0
  Missing (check these): ['cls_token']
  Gradient checkpointing enabled
  Total trainable params: 630,812,198
               head:       48,678 params, lr=1.00e-03
            block_0:   19,677,440 params, lr=7.89e-10
            block_1:   19,677,440 params, lr=1.13e-09
            block_2:   19,677,440 params, lr=1.61e-09
            block_3:   19,677,440 params, lr=2.30e-09
            block_4:   19,677,440 params, lr=3.29e-09
            block_5:   19,677,440 params, lr=4.69e-09
            block_6:   19,677,440 params, lr=6.71e-09
            block_7:   19,677,440 params, lr=9.58e-09
            block_8:   19,677,440 params, lr=1.37e-08
            block_9:   19,677,440 params, lr=1.95e-08
           block_10:   19,677,440 params, lr=2.79e-08
           block_11:   19,677,440 params, lr=3.99e-08
           block_12:   19,677,440 params, lr=5.70e-08

## Training

In [3]:
# Training
train_paths, train_labels, class_names = load_split(SPLITS_DIR / "train_split.json")
val_paths, val_labels, _ = load_split(SPLITS_DIR / "val_split.json")
test_paths, test_labels, _ = load_split(SPLITS_DIR / "test_split.json")

train_tf = get_pretrain_transform()
eval_tf  = get_eval_transform()

train_ds = PlantVillageDataset(train_paths, train_labels, transform=train_tf)
val_ds   = PlantVillageDataset(val_paths, val_labels, transform=eval_tf)
test_ds  = PlantVillageDataset(test_paths, test_labels, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=FT_BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=FT_BATCH_SIZE * 2, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=FT_BATCH_SIZE * 2, shuffle=False,
                          num_workers=4, pin_memory=True)

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(param_groups, weight_decay=FT_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=FT_EPOCHS)
scaler = GradScaler(enabled=torch.cuda.is_available())
es = EarlyStopping(patience=FT_PATIENCE)

if WANDB_ENTITY:
    os.environ["WANDB__SERVICE_WAIT"] = "10"
    os.environ["WANDB_DISABLED"] = "true"
    try:
        wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                   name="B4-IJEPA-FullFT", reinit=True, config=hyperparams)
    except Exception:
        print("WandB init failed — training without logging")
        WANDB_ENTITY = False


print(f"\nTraining B4 I-JEPA Full Fine-Tune (100% labels)...")
print(f"  Batch: {FT_BATCH_SIZE} x {FT_GRAD_ACCUM} accum = {FT_BATCH_SIZE * FT_GRAD_ACCUM} effective")

t0 = time.time()
for epoch in range(FT_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion,
                                 optimiser, scaler, device,
                                 grad_accum_steps=FT_GRAD_ACCUM)
    val_result = evaluate(model, val_loader, device)
    scheduler.step()

    if WANDB_ENTITY:
        wandb.log({"train_loss": train_loss,
                    "val_macro_f1": val_result["macro_f1"],
                    "val_accuracy": val_result["accuracy"], "epoch": epoch})

    print(f"  Epoch {epoch+1:3d} | loss={train_loss:.4f} | "
          f"val_F1={val_result['macro_f1']:.4f} | val_acc={val_result['accuracy']:.4f}")

    es.step(val_result["macro_f1"], model)
    if es.stopped:
        print(f"  Early stop at epoch {epoch+1}")
        break

elapsed = time.time() - t0
es.load_best(model)
print(f"  Training time: {elapsed:.0f}s ({elapsed/3600:.1f}h)")

# Test evaluation
test_result = evaluate(model, test_loader, device)
print(f"\n  B4 Test macro-F1:  {test_result['macro_f1']:.4f}")
print(f"  B4 Test accuracy:  {test_result['accuracy']:.4f}")

if WANDB_ENTITY:
    wandb.log({"test_macro_f1": test_result["macro_f1"],
               "test_accuracy": test_result["accuracy"]})
    wandb.finish()

b4_aggregate = {
    "baseline": "B4", "model": "I-JEPA ViT-H/14 Full Fine-Tune",
    "macro_f1": test_result["macro_f1"],
    "accuracy": test_result["accuracy"],
    "per_class_f1": test_result["per_class_f1"],
    "training_time_s": elapsed,
    "best_val_f1": float(es.best_score),
    "trainable_params": n_trainable,
}
save_results(b4_aggregate, BASELINES_DIR / "B4_aggregate.json")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(
    np.array(test_result["confusion_matrix"]),
    class_names, FIGURES_DIR / "B4_confusion_matrix.png",
    title=f"B4 I-JEPA Full FT | F1={test_result['macro_f1']:.3f}"
)

del model, optimiser, scheduler, scaler
torch.cuda.empty_cache()
print("\nB4 complete.")

C:\Users\muhha\AppData\Local\Temp\ipykernel_2436\152026848.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\muhha\_netrc.
wandb: Currently logged in as: muh-haleef02 (muh-haleef02-inform) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB init failed — training without logging

Training B4 I-JEPA Full Fine-Tune (100% labels)...
  Batch: 16 x 4 accum = 64 effective


KeyboardInterrupt: 

## Critical Analysis

B4 establishes the full fine-tuning upper bound:

1. **Ceiling performance** — B4 should achieve 96-99% macro-F1 at 100% labels. This is the target PEFT methods aim to approach.

2. **RQ2 anchor** — If LoRA achieves 97% F1 with 0.3M params vs B4's 97.5% with 632M params, that's a compelling efficiency argument.

3. **Training cost** — Document GPU hours and peak VRAM. The cost difference between B4 and PEFT methods is a key practical finding.

4. **Overfitting risk** — ViTs can overfit PlantVillage at full data. If val F1 peaks early and then degrades, this motivates parameter-efficient approaches.